# 6.1 · Authorization: OAuth 2.1 in MCP

**授权：MCP 中的 OAuth 2.1**

> **After this lesson you can:** Walk an MCP client through the full OAuth 2.1 flow — 401 challenge, metadata discovery, registration, PKCE, resource-bound tokens, iss validation and scope step-up — and protect a server that validates audience and scope.

Official reading: [Understanding authorization in MCP](https://modelcontextprotocol.io/docs/2026-07-28/tutorials/security/authorization) · [Spec: Authorization](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization/index) · [Spec: Authorization server discovery](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization/authorization-server-discovery) · [Spec: Client registration](https://modelcontextprotocol.io/specification/2026-07-28/basic/authorization/client-registration)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. PKCE by hand
PKCE is just a random secret and its SHA-256 hash. The client sends the **challenge** in the browser redirect (which may leak) and keeps the **verifier** until the back-channel token request.

In [ ]:
import base64
import hashlib
import json
import logging
import secrets

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpx2").setLevel(logging.WARNING)
logging.getLogger("mcp").setLevel(logging.ERROR)


def b64url(raw: bytes) -> str:
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode()


def make_pkce() -> tuple[str, str]:
    verifier = b64url(secrets.token_bytes(48))              # 64 chars, within the 43..128 allowed
    challenge = b64url(hashlib.sha256(verifier.encode()).digest())
    return verifier, challenge


verifier, challenge = make_pkce()
print("code_verifier :", verifier, f"({len(verifier)} chars, stays on the client)")
print("code_challenge:", challenge, "(goes in the authorization URL)")
print("method        : S256")
# the AS later recomputes the hash from the verifier the client presents:
print("verifies?     :", b64url(hashlib.sha256(verifier.encode()).digest()) == challenge)

### 2. A toy authorization server
A real deployment uses Keycloak, Entra ID, Auth0… Here a ~60-line Starlette app plays the AS so you can see every field. It publishes RFC 8414 metadata (with `code_challenge_methods_supported` and `authorization_response_iss_parameter_supported`), auto-approves the user "alice" at `/authorize` (standing in for the login + consent page), enforces exact redirect-URI matching and PKCE at `/token`, and stamps the requested `resource` into the token's `aud`. Tokens are HMAC-signed JWTs — fine for a lesson, **never hand-roll this in production**.

In [ ]:
import hmac
import socket
import threading
import time
from urllib.parse import parse_qs, urlencode, urlparse

import uvicorn
from starlette.applications import Starlette
from starlette.responses import JSONResponse, RedirectResponse
from starlette.routing import Route


def free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def serve(app, port):
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.02)
    return server


AS_PORT, MCP_PORT = free_port(), free_port()
ISSUER = f"http://127.0.0.1:{AS_PORT}"
RESOURCE = f"http://127.0.0.1:{MCP_PORT}/mcp"          # the MCP server's canonical URI
SIGNING_KEY = secrets.token_bytes(32)
REDIRECT_URI = "http://127.0.0.1:33418/callback"


def mint_jwt(claims: dict) -> str:
    head = b64url(json.dumps({"alg": "HS256", "typ": "JWT"}).encode())
    body = b64url(json.dumps(claims).encode())
    sig = b64url(hmac.new(SIGNING_KEY, f"{head}.{body}".encode(), hashlib.sha256).digest())
    return f"{head}.{body}.{sig}"


def read_jwt(token: str) -> dict | None:
    """Return the claims if the signature is valid, else None."""
    try:
        head, body, sig = token.split(".")
        good = b64url(hmac.new(SIGNING_KEY, f"{head}.{body}".encode(), hashlib.sha256).digest())
        if not hmac.compare_digest(sig, good):
            return None
        return json.loads(base64.urlsafe_b64decode(body + "=" * (-len(body) % 4)))
    except ValueError:
        return None


CLIENTS = {"notes-cli": {"redirect_uris": [REDIRECT_URI]}}   # pre-registered clients
CODES: dict[str, dict] = {}
REFRESH: dict[str, dict] = {}


async def as_metadata(request):
    return JSONResponse({
        "issuer": ISSUER,
        "authorization_endpoint": f"{ISSUER}/authorize",
        "token_endpoint": f"{ISSUER}/token",
        "response_types_supported": ["code"],
        "grant_types_supported": ["authorization_code", "refresh_token"],
        "code_challenge_methods_supported": ["S256"],
        "token_endpoint_auth_methods_supported": ["none"],
        "scopes_supported": ["notes:read", "notes:write"],
        "authorization_response_iss_parameter_supported": True,
    })


async def authorize(request):
    q = request.query_params
    client = CLIENTS.get(q.get("client_id"))
    if client is None or q.get("redirect_uri") not in client["redirect_uris"]:   # exact match only
        return JSONResponse({"error": "invalid_request", "error_description": "unknown client or redirect_uri"}, 400)
    if q.get("code_challenge_method") != "S256" or not q.get("code_challenge"):
        return JSONResponse({"error": "invalid_request", "error_description": "PKCE S256 required"}, 400)
    # ... a real AS shows a login + consent page here; we auto-approve user "alice" ...
    code = secrets.token_urlsafe(16)
    CODES[code] = {"client_id": q["client_id"], "redirect_uri": q["redirect_uri"], "challenge": q["code_challenge"],
                   "scope": q.get("scope", ""), "resource": q.get("resource"), "sub": "alice"}
    back = {"code": code, "state": q.get("state", ""), "iss": ISSUER}             # RFC 9207 iss
    return RedirectResponse(f"{q['redirect_uri']}?{urlencode(back)}", status_code=302)


def issue_tokens(sub, client_id, scope, resource, ttl=300):
    now = int(time.time())
    access = mint_jwt({"iss": ISSUER, "sub": sub, "aud": resource, "client_id": client_id,
                       "scope": scope, "iat": now, "exp": now + ttl})
    refresh = secrets.token_urlsafe(24)
    REFRESH[refresh] = {"sub": sub, "client_id": client_id, "scope": scope, "resource": resource}
    return {"access_token": access, "token_type": "Bearer", "expires_in": ttl, "refresh_token": refresh, "scope": scope}


async def token(request):
    form = await request.form()
    if form.get("grant_type") == "authorization_code":
        rec = CODES.pop(form.get("code"), None)                                   # codes are single-use
        if rec is None or rec["client_id"] != form.get("client_id") or rec["redirect_uri"] != form.get("redirect_uri"):
            return JSONResponse({"error": "invalid_grant"}, 400)
        if b64url(hashlib.sha256(form.get("code_verifier", "").encode()).digest()) != rec["challenge"]:
            return JSONResponse({"error": "invalid_grant", "error_description": "PKCE verification failed"}, 400)
        if form.get("resource") != rec["resource"]:
            return JSONResponse({"error": "invalid_target"}, 400)
        return JSONResponse(issue_tokens(rec["sub"], rec["client_id"], rec["scope"], rec["resource"]))
    if form.get("grant_type") == "refresh_token":
        rec = REFRESH.pop(form.get("refresh_token"), None)                         # rotation: old one dies
        if rec is None:
            return JSONResponse({"error": "invalid_grant", "error_description": "unknown or reused refresh token"}, 400)
        return JSONResponse(issue_tokens(rec["sub"], rec["client_id"], rec["scope"], rec["resource"]))
    return JSONResponse({"error": "unsupported_grant_type"}, 400)


as_app = Starlette(routes=[Route("/.well-known/oauth-authorization-server", as_metadata),
                           Route("/authorize", authorize),
                           Route("/token", token, methods=["POST"])])
as_server = serve(as_app, AS_PORT)
print("authorization server:", ISSUER)

### 3. A protected MCP server with the SDK
The SDK does the resource-server plumbing: given `token_verifier=` and `auth=AuthSettings(...)` it publishes PRM, answers unauthenticated requests with `401` + `resource_metadata`, and — with `validate_token_resource=True` — rejects tokens whose `AccessToken.resource` is not `resource_server_url`. `required_scopes` apply to *every* request, so per-tool scopes need your own check: a small ASGI wrapper reads the 2026-07-28 `Mcp-Method` / `Mcp-Name` headers and answers `403 insufficient_scope` for `delete_note` unless the token has `notes:write`.

In [ ]:
from mcp.server import MCPServer
from mcp.server.auth.provider import AccessToken
from mcp.server.auth.settings import AuthSettings


class HmacTokenVerifier:
    """Signature + issuer + expiry. The SDK then compares `resource` (the aud) with resource_server_url."""

    async def verify_token(self, token: str) -> AccessToken | None:
        claims = read_jwt(token)
        if not claims or claims.get("iss") != ISSUER or claims.get("exp", 0) < time.time():
            return None
        return AccessToken(token=token, client_id=claims["client_id"], scopes=claims["scope"].split(),
                           expires_at=claims["exp"], resource=claims["aud"], subject=claims["sub"])


notes_server = MCPServer(
    "notes",
    token_verifier=HmacTokenVerifier(),
    auth=AuthSettings(issuer_url=ISSUER, resource_server_url=RESOURCE,
                      required_scopes=["notes:read"], validate_token_resource=True),
)
NOTES = {"n1": "buy milk", "n2": "renew passport"}


@notes_server.tool()
def list_notes() -> list[str]:
    """List the ids of the user's notes."""
    return sorted(NOTES)


@notes_server.tool()
def delete_note(note_id: str) -> str:
    """Delete one note (needs notes:write)."""
    NOTES.pop(note_id, None)
    return f"deleted {note_id}"


PRM_URL = f"http://127.0.0.1:{MCP_PORT}/.well-known/oauth-protected-resource/mcp"
TOOL_SCOPES = {"delete_note": "notes:write"}
inner_app = notes_server.streamable_http_app()


async def scope_gate(scope, receive, send):
    """Per-tool scope check that emits a spec-shaped 403 challenge."""
    if scope["type"] == "http":
        headers = {k.decode().lower(): v.decode() for k, v in scope["headers"]}
        needed = TOOL_SCOPES.get(headers.get("mcp-name")) if headers.get("mcp-method") == "tools/call" else None
        auth = headers.get("authorization", "")
        if needed and auth.lower().startswith("bearer "):
            claims = read_jwt(auth[7:]) or {}
            if needed not in claims.get("scope", "").split():
                challenge = (f'Bearer error="insufficient_scope", scope="{needed}", '
                             f'resource_metadata="{PRM_URL}", error_description="{needed} required"')
                await send({"type": "http.response.start", "status": 403,
                            "headers": [(b"content-type", b"application/json"),
                                        (b"www-authenticate", challenge.encode())]})
                await send({"type": "http.response.body", "body": b'{"error": "insufficient_scope"}'})
                return
    await inner_app(scope, receive, send)        # lifespan and everything else pass straight through


mcp_server = serve(scope_gate, MCP_PORT)
print("protected MCP server:", RESOURCE)

### 4. Step 1 — the 401 challenge
Call the server like a client that knows nothing. The interesting part is the `WWW-Authenticate` header.

In [ ]:
import re

import httpx

http = httpx.AsyncClient(trust_env=False)
MCP_HEADERS = {"Accept": "application/json, text/event-stream", "MCP-Protocol-Version": "2026-07-28",
               "Mcp-Method": "tools/list"}
LIST_TOOLS = {"jsonrpc": "2.0", "id": 1, "method": "tools/list",
              "params": {"_meta": {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
                                   "io.modelcontextprotocol/clientCapabilities": {}}}}


def parse_www_authenticate(value: str) -> dict:
    return dict(re.findall(r'(\w+)="([^"]*)"', value or ""))


r = await http.post(RESOURCE, json=LIST_TOOLS, headers=MCP_HEADERS)
print("HTTP", r.status_code)
print("WWW-Authenticate:", r.headers["www-authenticate"])
challenge = parse_www_authenticate(r.headers["www-authenticate"])
print(json.dumps(challenge, indent=2))

Notice the SDK always adds `error="invalid_token"` and never a `scope=` on its 401. The spec only says the server **SHOULD** include `scope`; without it, the client falls back to PRM `scopes_supported`.

### 5. Steps 2–3 — Protected Resource Metadata, then the AS metadata
Follow the pointer, then build the AS well-known URLs in the order the spec mandates. Two checks the client **must** make: the `issuer` in the document equals the issuer we built the URL from, and PKCE support is advertised.

In [ ]:
prm = (await http.get(challenge["resource_metadata"])).json()
print("PRM:", json.dumps(prm, indent=2))


def as_metadata_urls(issuer: str) -> list[str]:
    u = urlparse(issuer)
    base, path = f"{u.scheme}://{u.netloc}", u.path.rstrip("/")
    if path:
        return [f"{base}/.well-known/oauth-authorization-server{path}",
                f"{base}/.well-known/openid-configuration{path}",
                f"{base}{path}/.well-known/openid-configuration"]
    return [f"{base}/.well-known/oauth-authorization-server", f"{base}/.well-known/openid-configuration"]


print("\ndiscovery order for a tenant issuer:", *as_metadata_urls("https://auth.example.com/tenant1"), sep="\n  ")

as_issuer = prm["authorization_servers"][0]
for url in as_metadata_urls(as_issuer):
    resp = await http.get(url)
    if resp.status_code == 200:
        asm = resp.json()
        print("\nfound AS metadata at", url)
        break

assert asm["issuer"] == as_issuer, "issuer mismatch -> MUST NOT use this metadata"
assert "S256" in asm.get("code_challenge_methods_supported", []), "no PKCE support -> MUST refuse"
print("issuer matches, S256 supported, iss-in-response advertised:",
      asm.get("authorization_response_iss_parameter_supported"))

### 6. Step 4 — choosing a registration mechanism
The priority order from the client-registration page, as a function. Our AS has no `registration_endpoint` and no CIMD support, but we were pre-registered as `notes-cli` — bound to this issuer.

In [ ]:
PRE_REGISTERED = {ISSUER: "notes-cli"}           # keyed by issuer: credentials are bound to their AS


def choose_registration(asm: dict, cimd_url: str | None) -> str:
    if asm["issuer"] in PRE_REGISTERED:
        return f"pre-registered client_id={PRE_REGISTERED[asm['issuer']]}"
    if cimd_url and asm.get("client_id_metadata_document_supported"):
        return f"CIMD: client_id={cimd_url}"
    if asm.get("registration_endpoint"):
        return "DCR (deprecated): POST registration_endpoint with application_type='native'"
    return "ask the user to enter client information"


print(choose_registration(asm, "https://app.example.com/oauth/client.json"))
print(choose_registration({"issuer": "https://other.example", "client_id_metadata_document_supported": True},
                          "https://app.example.com/oauth/client.json"))
print(choose_registration({"issuer": "https://legacy.example", "registration_endpoint": "https://legacy.example/register"}, None))

### 7. Step 5–6 — authorization request, callback, `state` and `iss`
We play the browser with `httpx` (no redirect following) and read the `Location` header the AS sends back. Scope selection: the 401 had no `scope`, so we take PRM `scopes_supported`. Before using the code we check `state` and apply the RFC 9207 table.

In [ ]:
def check_iss(as_meta: dict, recorded_issuer: str, iss: str | None) -> None:
    """The 2026-07-28 Authorization Response Validation table."""
    supported = as_meta.get("authorization_response_iss_parameter_supported") is True
    if iss is not None:
        if iss != recorded_issuer:                 # simple string comparison, no normalisation
            raise ValueError(f"iss mismatch: {iss!r} != {recorded_issuer!r} (possible mix-up attack)")
    elif supported:
        raise ValueError("iss missing although the AS advertises it")


verifier, challenge_s256 = make_pkce()
state = secrets.token_urlsafe(16)
pending = {"verifier": verifier, "state": state, "issuer": asm["issuer"]}      # recorded BEFORE redirecting
scope = challenge.get("scope") or " ".join(prm.get("scopes_supported", []))

auth_url = asm["authorization_endpoint"] + "?" + urlencode({
    "response_type": "code", "client_id": "notes-cli", "redirect_uri": REDIRECT_URI, "state": state,
    "code_challenge": challenge_s256, "code_challenge_method": "S256", "scope": scope, "resource": RESOURCE})
print("authorization URL:\n ", auth_url, "\n")

redirect = await http.get(auth_url)
callback = {k: v[0] for k, v in parse_qs(urlparse(redirect.headers["location"]).query).items()}
print("callback params:", json.dumps(callback, indent=2))

assert callback["state"] == pending["state"], "state mismatch -> discard"
check_iss(asm, pending["issuer"], callback.get("iss"))
print("state and iss OK")
for bad_iss in ["http://evil.example", ISSUER + "/", None]:
    try:
        check_iss(asm, pending["issuer"], bad_iss)
    except ValueError as exc:
        print("rejected:", exc)

The trailing-slash case is rejected on purpose: the spec forbids normalising before comparison.

### 8. Step 7 — redeem the code
The token request carries the `code_verifier` and the same `resource`. Decode the access token to see the audience binding.

In [ ]:
token_endpoint = asm["token_endpoint"]
form = {"grant_type": "authorization_code", "code": callback["code"], "redirect_uri": REDIRECT_URI,
        "client_id": "notes-cli", "code_verifier": pending["verifier"], "resource": RESOURCE}
tokens = (await http.post(token_endpoint, data=form)).json()
print({k: (v[:25] + "…" if isinstance(v, str) and len(v) > 25 else v) for k, v in tokens.items()})
print("access token claims:", json.dumps(read_jwt(tokens["access_token"]), indent=2))

replay = await http.post(token_endpoint, data=form)          # codes are single-use
print("replaying the same code ->", replay.status_code, replay.json())

### 9. Step 8 — call the server with the token
`Client(url)` builds its own HTTP client; to add headers, pass a transport built with `streamable_http_client(url, http_client=...)` (the SDK uses `httpx2`). The same token goes on **every** request.

In [ ]:
import httpx2

from mcp import Client
from mcp.client.streamable_http import streamable_http_client

access_token = tokens["access_token"]
async with httpx2.AsyncClient(headers={"Authorization": f"Bearer {access_token}"}, trust_env=False) as hc:
    async with Client(streamable_http_client(RESOURCE, http_client=hc)) as client:
        print("tools:", [t.name for t in (await client.list_tools()).tools])
        result = await client.call_tool("list_notes", {})
        print("list_notes ->", result.structured_content)

# the same thing on the wire
r = await http.post(RESOURCE, json=LIST_TOOLS, headers={**MCP_HEADERS, "Authorization": f"Bearer {access_token}"})
print("\nraw HTTP", r.status_code, r.headers["content-type"])
print(json.dumps(r.json()["result"]["tools"][0], indent=2)[:400])

### 10. Break it — tokens the server must refuse
Each of these is a valid-looking token. Only the audience-bound, untampered, unexpired one for *this* issuer works.

In [ ]:
now = int(time.time())
good = read_jwt(access_token)
candidates = {
    "correct token": access_token,
    "audience = another service": mint_jwt({**good, "aud": "https://payroll.example.com/mcp"}),
    "expired": mint_jwt({**good, "exp": now - 10}),
    "other issuer (same key!)": mint_jwt({**good, "iss": "https://other-tenant.example"}),
    "tampered scope": access_token.split(".")[0] + "." + b64url(json.dumps({**good, "scope": "notes:read notes:write"}).encode()) + "." + access_token.split(".")[2],
}
for label, tok in candidates.items():
    r = await http.post(RESOURCE, json=LIST_TOOLS, headers={**MCP_HEADERS, "Authorization": f"Bearer {tok}"})
    print(f"{label:28} -> {r.status_code}")

r = await http.post(RESOURCE + f"?access_token={access_token}", json=LIST_TOOLS, headers=MCP_HEADERS)
print(f"{'token in query string':28} -> {r.status_code}  (MUST NOT be used; the server ignores it)")

### 11. Scope challenge and step-up, by hand
Our token only has `notes:read`. Calling `delete_note` gets the runtime **403** with the needed scope. The client re-authorizes with the **union** of old and new scopes and retries.

In [ ]:
call = {"jsonrpc": "2.0", "id": 2, "method": "tools/call",
        "params": {"name": "delete_note", "arguments": {"note_id": "n2"},
                   "_meta": LIST_TOOLS["params"]["_meta"]}}
call_headers = {**MCP_HEADERS, "Mcp-Method": "tools/call", "Mcp-Name": "delete_note"}

r = await http.post(RESOURCE, json=call, headers={**call_headers, "Authorization": f"Bearer {access_token}"})
print("HTTP", r.status_code, "\nWWW-Authenticate:", r.headers["www-authenticate"])
step = parse_www_authenticate(r.headers["www-authenticate"])

granted = set(tokens["scope"].split())
wanted = " ".join(sorted(granted | set(step["scope"].split())))      # union, so notes:read is not lost
print("re-authorizing with scope =", repr(wanted))


async def authorize_and_redeem(scope: str) -> dict:
    v, c = make_pkce()
    s = secrets.token_urlsafe(16)
    loc = (await http.get(asm["authorization_endpoint"] + "?" + urlencode({
        "response_type": "code", "client_id": "notes-cli", "redirect_uri": REDIRECT_URI, "state": s,
        "code_challenge": c, "code_challenge_method": "S256", "scope": scope, "resource": RESOURCE}))).headers["location"]
    cb = {k: x[0] for k, x in parse_qs(urlparse(loc).query).items()}
    assert cb["state"] == s
    check_iss(asm, asm["issuer"], cb.get("iss"))
    return (await http.post(asm["token_endpoint"], data={
        "grant_type": "authorization_code", "code": cb["code"], "redirect_uri": REDIRECT_URI,
        "client_id": "notes-cli", "code_verifier": v, "resource": RESOURCE})).json()


tokens = await authorize_and_redeem(wanted)
r = await http.post(RESOURCE, json=call, headers={**call_headers, "Authorization": f"Bearer {tokens['access_token']}"})
print("retry ->", r.status_code, r.json()["result"]["content"][0]["text"])

### 12. Refresh-token rotation
Access tokens are short-lived; a refresh token gets a new one without a browser. Our AS rotates refresh tokens (the spec requires that for public clients), so a stolen, already-used one is dead.

In [ ]:
old_refresh = tokens["refresh_token"]
fresh = (await http.post(token_endpoint, data={"grant_type": "refresh_token", "refresh_token": old_refresh,
                                               "client_id": "notes-cli", "resource": RESOURCE})).json()
print("new access token aud/scope:", read_jwt(fresh["access_token"])["aud"], "|", fresh["scope"])
print("refresh token rotated:", fresh["refresh_token"] != old_refresh)
reuse = await http.post(token_endpoint, data={"grant_type": "refresh_token", "refresh_token": old_refresh,
                                              "client_id": "notes-cli", "resource": RESOURCE})
print("reusing the old refresh token ->", reuse.status_code, reuse.json())

### 13. Let the SDK do all of it
`mcp.client.auth.OAuthClientProvider` is an `httpx2.Auth` that runs the whole flow on a 401 — PRM discovery with fallbacks, AS metadata with issuer validation, registration, PKCE, `state` + `iss` checks, the token request with `resource` — and the **step-up** on a `403 insufficient_scope`. You supply storage (here: in memory, pre-registered and bound to our issuer) and the two browser hooks. An event hook prints every HTTP exchange.

In [ ]:
from mcp.client.auth import AuthorizationCodeResult, OAuthClientProvider
from mcp.shared.auth import OAuthClientInformationFull, OAuthClientMetadata

NOTES["n2"] = "renew passport"


class MemoryStorage:
    def __init__(self):
        self.tokens = None
        self.client = OAuthClientInformationFull(client_id="notes-cli", redirect_uris=[REDIRECT_URI],
                                                 token_endpoint_auth_method="none", issuer=ISSUER)

    async def get_tokens(self): return self.tokens
    async def set_tokens(self, tokens): self.tokens = tokens
    async def get_client_info(self): return self.client
    async def set_client_info(self, info): self.client = info


browser = {}


async def redirect_handler(url: str) -> None:          # a desktop app would open the system browser
    print("    [browser] user approves at", url.split("?")[0])
    async with httpx.AsyncClient(trust_env=False) as b:
        browser["location"] = (await b.get(url)).headers["location"]


async def callback_handler() -> AuthorizationCodeResult:  # ...and listen on the redirect URI
    q = {k: v[0] for k, v in parse_qs(urlparse(browser["location"]).query).items()}
    return AuthorizationCodeResult(code=q["code"], state=q.get("state"), iss=q.get("iss"))


provider = OAuthClientProvider(
    server_url=RESOURCE,
    client_metadata=OAuthClientMetadata(client_name="notes-cli", redirect_uris=[REDIRECT_URI],
                                        grant_types=["authorization_code", "refresh_token"],
                                        token_endpoint_auth_method="none"),
    storage=MemoryStorage(), redirect_handler=redirect_handler, callback_handler=callback_handler)


async def log_exchange(response):
    req = response.request
    print(f"  {req.method:4} {req.url.path:48} -> {response.status_code}  {req.headers.get('mcp-method', '')}")


async with httpx2.AsyncClient(auth=provider, trust_env=False, event_hooks={"response": [log_exchange]}) as hc:
    async with Client(streamable_http_client(RESOURCE, http_client=hc)) as client:
        print("list_notes ->", (await client.call_tool("list_notes", {})).structured_content)
        print("granted scope:", provider.context.current_tokens.scope, "\n")
        r = await client.call_tool("delete_note", {"note_id": "n2"})
        print("delete_note ->", r.content[0].text)
        print("granted scope:", provider.context.current_tokens.scope)

Read the log: one 401, PRM, AS metadata, browser, `/token`, retry; later a 403, browser again with the **union** of scopes, `/token`, retry. Two SDK quirks worth knowing: its server-side 403 (from `required_scopes`) has no `scope=` parameter, and the client provider does not itself refuse an AS whose metadata omits `code_challenge_methods_supported` — check that yourself if your AS might.

In [ ]:
await http.aclose()
mcp_server.should_exit = True
as_server.should_exit = True
print("servers stopped")

### Your turn
1. Make the AS "forget" PKCE: remove `code_challenge_methods_supported` from `as_metadata`, re-run section 5, and write the client-side check that refuses to proceed. Then confirm the SDK provider in section 13 does *not* refuse — and add a guard.
2. Change the toy AS to put `"aud": "https://api.example.com"` in every token regardless of `resource` (the Keycloak tutorial's fixed-audience shortcut). Which section fails, and what would a production AS do instead?
3. Make `scope_gate` return **two** missing scopes for `delete_note` (`notes:write audit:write`) and confirm the SDK requests the union of all three in one step-up, not two.
4. Add a second entry to PRM `authorization_servers`. Where in the flow does the client choose, and why must its stored `client_id` for the first AS never be sent to the second?